# LoRA verifier: a small code model that reads the issue and a patch, and predicts whether the hidden tests pass

Runs on a Colab GPU (L4 or A100, about an hour). It trains **Qwen2.5-Coder-1.5B** with LoRA as a binary classifier on
`data/verifier.jsonl` (written by `sweverifier export`), **out of fold**: the five folds are grouped by issue, and every
candidate is scored by the model trained on the four folds that do not contain its issue. The scores go to
`results/lora_scores.jsonl`; `sweverifier verify` then compares this verifier with the others.

In [ ]:
!pip -q install "transformers>=4.46" "peft>=0.13" accelerate scikit-learn
!pip uninstall -q -y torchao  # Colab's preinstalled torchao is too old for peft, which refuses to load next to it
!git clone -q https://github.com/LacosteEvan/swe-verifier.git
%cd swe-verifier

In [ ]:
import json

import numpy as np
import torch
from peft import LoraConfig, get_peft_model
from sklearn.metrics import roc_auc_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
MAX_ISSUE, MAX_PATCH = 1500, 1500   # tokens kept from each, so a candidate fits in about 3,100 tokens
EPOCHS, LR, ACCUM = 2, 1e-4, 8

rows = [json.loads(l) for l in open("data/verifier.jsonl")]
print(len(rows), "candidates,", sum(r["label"] for r in rows), "resolved,", len({r["instance_id"] for r in rows}), "issues")
tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.pad_token or tok.eos_token

def encode(r):
    issue = tok(r["issue"], add_special_tokens=False)["input_ids"][:MAX_ISSUE]
    patch = tok(r["patch"] or "(empty patch)", add_special_tokens=False)["input_ids"][:MAX_PATCH]
    head = tok(f"Repository: {r['repo']}\nIssue:\n", add_special_tokens=False)["input_ids"]
    mid = tok("\n\nCandidate patch:\n", add_special_tokens=False)["input_ids"]
    tail = tok("\n\nDoes this patch resolve the issue?", add_special_tokens=False)["input_ids"]
    return head + issue + mid + patch + tail

for r in rows:
    r["ids"] = encode(r)
print("longest input:", max(len(r["ids"]) for r in rows), "tokens")

In [ ]:
def train_fold(train):
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=1, torch_dtype=torch.bfloat16)
    model.config.pad_token_id = tok.pad_token_id
    model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="SEQ_CLS",
                                             target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]))
    model.cuda().train()
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
    pos = sum(r["label"] for r in train) / len(train)
    weight = torch.tensor((1 - pos) / pos).cuda()   # the resolved candidates are the minority
    steps = EPOCHS * len(train) // ACCUM
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / 10) * max(0.0, 1 - s / steps))
    rng = np.random.default_rng(0)
    for epoch in range(EPOCHS):
        for i, j in enumerate(rng.permutation(len(train))):
            r = train[j]
            logit = model(input_ids=torch.tensor([r["ids"]]).cuda()).logits.float().squeeze()
            loss = torch.nn.functional.binary_cross_entropy_with_logits(
                logit, torch.tensor(float(r["label"])).cuda(), pos_weight=weight) / ACCUM
            loss.backward()
            if (i + 1) % ACCUM == 0:
                opt.step(); sched.step(); opt.zero_grad()
    return model.eval()

@torch.no_grad()
def predict(model, rs):
    return [torch.sigmoid(model(input_ids=torch.tensor([r["ids"]]).cuda()).logits.float()).item() for r in rs]

In [ ]:
scores = {}
for fold in sorted({r["fold"] for r in rows}):
    train = [r for r in rows if r["fold"] != fold]
    test = [r for r in rows if r["fold"] == fold]
    model = train_fold(train)
    for r, p in zip(test, predict(model, test)):
        scores[(r["instance_id"], r["sample"])] = p
    y, s = [r["label"] for r in test], [scores[(r["instance_id"], r["sample"])] for r in test]
    print(f"fold {fold}: {len(train)} train, {len(test)} test, AUC {roc_auc_score(y, s) if len(set(y)) == 2 else float('nan'):.3f}")
    del model; torch.cuda.empty_cache()

y = [r["label"] for r in rows]; s = [scores[(r["instance_id"], r["sample"])] for r in rows]
print("out-of-fold AUC:", round(roc_auc_score(y, s), 3))

In [ ]:
import os

os.makedirs("results", exist_ok=True)
with open("results/lora_scores.jsonl", "w") as f:
    for (iid, k), p in sorted(scores.items()):
        f.write(json.dumps({"instance_id": iid, "sample": k, "p": p, "model": MODEL}) + "\n")

# best-of-N with the LoRA score, against picking a candidate at random
by = {}
for r in rows:
    by.setdefault(r["instance_id"], []).append((scores[(r["instance_id"], r["sample"])], r["label"]))
top = lambda c: [l for p, l in c if p == max(q for q, _ in c)]   # ties shared, never broken by the label
picked = np.mean([np.mean(top(c)) for c in by.values()])
random = np.mean([np.mean([l for _, l in c]) for c in by.values()])
print(f"resolved: random candidate {random:.1%}, LoRA verifier's pick {picked:.1%}")

Download `results/lora_scores.jsonl` into the repository's `results/` folder, then run `sweverifier verify` and
`python scripts/make_report.py`: the LoRA verifier joins the comparison table and the chart.